# **EV Battery Training Model Notebook**

## Objectives

"Fetch data from Kaggle and save as Data Frame", or "engineer features for modelling"

## Inputs

Our Kaggle dataset and the experience of the exploration we learnt from the visuals

## Outputs

A trained model that will predict the 'State of Health' score of a battery

## Additional Comments

* Introduce 



---

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

# Section 1

Section 1 content

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "ev_battery_health.csv"
file_dictionary = "data_dictionary .csv"
# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "vinay24baghira/ev-battery-health-and-degradation-dataset",
  file_path
)

ddic = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "vinay24baghira/ev-battery-health-and-degradation-dataset",
  file_dictionary
)




c:\scripts\codeInst_projects\Project2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

# Section 2

drop missing data

In [2]:
##We can not have these rows in our data because there is not enough time/count data in the reading if all three are missing. 5 rows removed as we can't use those readings.  Forward thinking is time will be important and we can still substitute battery_age_days and cycle count to estimate when a reading happened
columns_to_check = ['observation_date', 'battery_age_days', 'cycle_count']

# Record starting shape
initial_shape = df.shape

# Drop rows where all three columns are null
df = df.dropna(subset=columns_to_check, how='all')

# Print changes
print(f"Original shape: {initial_shape}")
print(f"New shape: {df.shape}")
print(f"Removed rows: {initial_shape[0] - df.shape[0]}")

Original shape: (50000, 35)
New shape: (49995, 35)
Removed rows: 5


---

---

In [3]:
def get_non_null_value(battery_id, col_name):
    """
    Looks up a non-null value for a given battery_id and column name from df.
    """
    values = df[df['battery_id'] == battery_id][col_name].dropna()
    if not values.empty:
        return values.iloc[0]
    return np.nan

# Group the DataFrame by 'battery_id' to build an efficient lookup dictionary for each specified column
print("Populating gaps in the dataset for specified columns...")

# Specific columns to fill, these rows related to the battery pack and not the reading. So just for completelness.
columns_to_fill = ["chemistry", "manufacturer", "vehicle_model", "pack_capacity_kwh", "manufacture_date", "climate_zone"]

# Iterate and apply imputation using other rows of the same battery_id
for col in columns_to_fill:
    if col in df.columns:
        # Create a lookup mapping battery_id to its first non-null value for this column
        lookup_map = df.groupby('battery_id')[col].first()

        # Get indices where the column is null
        null_mask = df[col].isnull()

        # Map the battery_id of null rows to the lookup map
        imputed_values = df.loc[null_mask, 'battery_id'].map(lookup_map)

        # Fill the missing values
        df.loc[null_mask, col] = df.loc[null_mask, col].fillna(imputed_values)

# Verify new missing value counts for the selected columns
remaining_missing = df[columns_to_fill].isnull().sum()
print("\nRemaining missing data counts for targeted columns:")
print(remaining_missing)

Populating gaps in the dataset for specified columns...

Remaining missing data counts for targeted columns:
chemistry            0
manufacturer         0
vehicle_model        0
pack_capacity_kwh    0
manufacture_date     0
climate_zone         0
dtype: int64


In [4]:
# Define columns for each table, there are two tables if we think of the flat table as containing two objects, the battery and its readings.
battery_cols = ["battery_id", "chemistry", "manufacturer", "vehicle_model", "pack_capacity_kwh", "manufacture_date", "climate_zone"]
reading_cols = ["observation_id", "battery_id", "observation_date", "battery_age_days", "cycle_count", "avg_daily_distance_km", "avg_charging_power_kw", "fast_charge_ratio", "avg_depth_of_discharge_pct", "avg_state_of_charge_pct", "charge_cycles_per_month", "total_energy_throughput_kwh", "avg_ambient_temp_c", "humidity_pct", "altitude_m", "internal_resistance_mohm", "voltage_avg_v", "current_avg_a", "charge_time_avg_min", "peak_temp_during_fast_charge_c", "capacity_fade_pct", "service_count", "last_service_date", "bms_firmware_version", "recall_flag", "state_of_health_pct", "remaining_useful_life_cycles", "failure_risk_label", "degradation_category"]

# Create BatteryTable with unique battery_id entries
BatteryTable = df[battery_cols].drop_duplicates(subset=["battery_id"]).reset_index(drop=True)

# Create ReadingTable with unique observation_id entries
# Note: battery_id is included as a foreign key to maintain the relationship
ReadingTable = df[reading_cols].drop_duplicates(subset=["observation_id"]).reset_index(drop=True)

# 1. Ensure observation_date is in datetime64 format ready for sort
ReadingTable['observation_date'] = pd.to_datetime(ReadingTable['observation_date'])

# Print the count of missing cycle_count values BEFORE the operation
initial_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values BEFORE interpolation: {initial_missing}")

# 2. Sort the table chronologically by battery_id and observation_date
ReadingTable = ReadingTable.sort_values(by=['battery_id', 'observation_date']).reset_index(drop=True)

# 3 & 4. Group by battery_id, perform linear interpolation, and apply ffill/bfill for boundary cases
def interpolate_battery_cycles(group):
    # Interpolate over the sorted sequence within each battery group
    group['cycle_count'] = group['cycle_count'].interpolate(method='linear').ffill().bfill()
    return group

# Apply the group-by operation using include_groups=False to avoid DeprecationWarnings
#ReadingTable = ReadingTable.groupby('battery_id', group_keys=False).apply(interpolate_battery_cycles, include_groups=False)

# 5. Print the count of missing cycle_count values AFTER the operation
final_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values AFTER interpolation: {final_missing}")

Missing 'cycle_count' values BEFORE interpolation: 2264
Missing 'cycle_count' values AFTER interpolation: 2264


In [5]:
ReadingTable.head()

,observation_id,battery_id,observation_date,battery_age_days,cycle_count,avg_daily_distance_km,avg_charging_power_kw,fast_charge_ratio,avg_depth_of_discharge_pct,avg_state_of_charge_pct,...,peak_temp_during_fast_charge_c,capacity_fade_pct,service_count,last_service_date,bms_firmware_version,recall_flag,state_of_health_pct,remaining_useful_life_cycles,failure_risk_label,degradation_category
0,OBS_000001,BATT_00001,2022-04-11,68.0,56.0,14.3,23.2,0.236,49.9,68.7,...,22.7,2.41,0.0,NaN,BMS_v2.3,0.0,97.59,884,Low,Normal
1,OBS_000002,BATT_00001,2022-05-02,89.0,79.0,59.3,10.6,0.247,59.1,64.1,...,30.7,1.75,0.0,NaN,BMS_v3.0,0.0,98.25,1051,Low,Normal
2,OBS_000003,BATT_00001,2022-05-18,105.0,94.0,60.8,32.1,0.232,52.9,71.5,...,26.2,0.60,0.0,NaN,BMS_v3.2,0.0,99.40,1396,Low,Normal
3,OBS_000004,BATT_00001,2023-01-26,358.0,330.0,11.3,33.8,0.267,65.7,67.7,...,24.5,6.11,1.0,2022-10-06,BMS_v2.3,NaN,93.89,1252,Low,Normal
4,OBS_000005,BATT_00001,2023-02-28,391.0,340.0,46.0,31.6,0.267,47.4,75.3,...,19.3,4.50,2.0,2022-07-13,BMS_v2.3,0.0,95.50,683,Low,Normal


In [6]:
# 1. Ensure observation_date is in datetime64 format ready for sort
ReadingTable['observation_date'] = pd.to_datetime(ReadingTable['observation_date'])

# Print the count of missing cycle_count values BEFORE the operation
initial_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values BEFORE interpolation: {initial_missing}")

# 2. Sort the table chronologically by battery_id and observation_date
ReadingTable = ReadingTable.sort_values(by=['battery_id', 'observation_date']).reset_index(drop=True)

# 3 & 4. Group by battery_id, perform linear interpolation, and apply ffill/bfill for boundary cases
def interpolate_battery_cycles(group):
    # Interpolate over the sorted sequence within each battery group
    group['cycle_count'] = group['cycle_count'].interpolate(method='linear').ffill().bfill()
    return group

# Apply the group-by operation using include_groups=False to avoid DeprecationWarnings
#ReadingTable = ReadingTable.groupby('battery_id', group_keys=False).apply(interpolate_battery_cycles, include_groups=False)

# 5. Print the count of missing cycle_count values AFTER the operation
final_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values AFTER interpolation: {final_missing}")

Missing 'cycle_count' values BEFORE interpolation: 2264
Missing 'cycle_count' values AFTER interpolation: 2264


In [7]:
display(BatteryTable.head())

,battery_id,chemistry,manufacturer,vehicle_model,pack_capacity_kwh,manufacture_date,climate_zone
0,BATT_01748,LMO,StormCell,Crossover-X2,48.5,2022-11-23,cold
1,BATT_00491,LFP,VoltaCore,Sedan-E1,61.5,2023-03-06,cold
2,BATT_00011,LFP,IonFlux,Van-E4,73.1,2025-06-24,cold
3,BATT_00651,NMC,Amperion,Sport-E9,75.6,2022-08-13,cold
4,BATT_02054,NCA,VoltaCore,Hatch-E3,92.7,2024-09-27,hot_arid


In [12]:
display(ReadingTable.head())


,observation_id,observation_date,battery_age_days,cycle_count,avg_daily_distance_km,avg_charging_power_kw,fast_charge_ratio,avg_depth_of_discharge_pct,avg_state_of_charge_pct,charge_cycles_per_month,...,capacity_fade_pct,service_count,last_service_date,bms_firmware_version,recall_flag,state_of_health_pct,remaining_useful_life_cycles,failure_risk_label,degradation_category,battery_id
0,OBS_000001,2022-04-11,68.0,56.0,14.3,23.2,0.236,49.9,68.7,11.8,...,2.41,0.0,NaN,BMS_v2.3,0.0,97.59,884,Low,Normal,BATT_00001
1,OBS_000002,2022-05-02,89.0,79.0,59.3,10.6,0.247,59.1,64.1,23.1,...,1.75,0.0,NaN,BMS_v3.0,0.0,98.25,1051,Low,Normal,BATT_00001
2,OBS_000003,2022-05-18,105.0,94.0,60.8,32.1,0.232,52.9,71.5,20.6,...,0.60,0.0,NaN,BMS_v3.2,0.0,99.40,1396,Low,Normal,BATT_00001
3,OBS_000004,2023-01-26,358.0,330.0,11.3,33.8,0.267,65.7,67.7,25.9,...,6.11,1.0,2022-10-06,BMS_v2.3,NaN,93.89,1252,Low,Normal,BATT_00001
4,OBS_000005,2023-02-28,391.0,340.0,46.0,31.6,0.267,47.4,75.3,19.9,...,4.50,2.0,2022-07-13,BMS_v2.3,0.0,95.50,683,Low,Normal,BATT_00001


In [ ]:
def impute_battery_readings(df):
    """
    Fills missing values in ReadingTable using a moving average of past observations within each battery_id group.
    """
    # Create a working copy of the DataFrame
    imputed_df = df.copy()

    # Target features specified by the user
    target_features = [
        'capacity_fade_pct',
        'internal_resistance_mohm',
        'total_energy_throughput_kwh',
        'fast_charge_ratio'
    ]

    # Step 1: Ensure date column is datetime format and sort chronologically within each battery_id
    imputed_df['observation_date'] = pd.to_datetime(imputed_df['observation_date'])
    imputed_df = imputed_df.sort_values(by=['battery_id', 'observation_date']).reset_index(drop=True)

    # Step 2: Define a function to perform rolling imputation on each battery group
    def process_group(group):
        for feature in target_features:
            if feature in group.columns:
                # Calculate moving average of past observations (closed on left to use only historical records)
                # We use a window size of 3 as a baseline (can adjust if required), min_periods=1
                rolling_avg = group[feature].shift(1).rolling(window=3, min_periods=1).mean()

                # Impute missing values using the moving average
                group[feature] = group[feature].fillna(rolling_avg)

                # Fallback ffill and bfill for initial boundary conditions or persistent NaNs
                group[feature] = group[feature].ffill().bfill()
        return group

    # Step 3: Apply the grouped operation strictly by battery_id while preserving battery_id column in the output DataFrame
    # We set group_keys=True so that the grouped keys are maintained, or we can restore the grouped index column.
    # Using group_keys=False and not using include_groups=False (keeping default or manually handling) ensures battery_id is retained.
    battery_ids = imputed_df['battery_id'].copy()

    imputed_df = imputed_df.groupby(
        'battery_id', group_keys=False
    ).apply(process_group)

    # Restore battery_id, aligned to the rows returned by apply
    imputed_df['battery_id'] = battery_ids.reindex(imputed_df.index)

    return imputed_df

# Run the imputation pipeline on ReadingTable
print("Missing counts before imputation:")
print(ReadingTable[['capacity_fade_pct', 'internal_resistance_mohm', 'total_energy_throughput_kwh', 'fast_charge_ratio']].isnull().sum())
ReadingTable = impute_battery_readings(ReadingTable)
print("\nMissing counts after imputation:")
print(ReadingTable[['capacity_fade_pct', 'internal_resistance_mohm', 'total_energy_throughput_kwh', 'fast_charge_ratio']].isnull().sum())

In [13]:
# 1. Merge ReadingTable with BatteryTable using battery_id
merged_df = pd.merge(
    ReadingTable,
    BatteryTable[['battery_id', 'chemistry', 'manufacturer']],  # Pull only the chemistry and manufacturer columns needed
    on='battery_id',
    how='inner',
)

In [14]:
import os
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

# 1. Define feature columns and target variable
features = [
    'internal_resistance_mohm',
    'cycle_count',
    'total_energy_throughput_kwh',
    'battery_age_days',
    'chemistry',
]
target = 'state_of_health_pct'

# Clean missing values
df_clean = merged_df[features + [target]].dropna().copy()
X = df_clean[features]
y = df_clean[target]

# 2. Perform Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Create Preprocessor for categorical 'chemistry'
num_cols = [
    'internal_resistance_mohm',
    'cycle_count',
    'total_energy_throughput_kwh',
    'battery_age_days',
]
cat_cols = ['chemistry']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
    ]
)

# 4. Build Pipeline (Preprocessing + Estimator)
pipeline = make_pipeline(
    preprocessor, RandomForestRegressor(n_estimators=100, random_state=42)
)

# 5. Train Model Pipeline
pipeline.fit(X_train, y_train)

# 6. Evaluate Model on Test Set
y_pred = pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print('--- Model Evaluation ---')
print(f'R² Score: {r2:.4f}')
print(f'Mean Absolute Error (MAE): {mae:.2f}%')

# 7. Safe File Saving Strategy
DATA_DIR = 'streamlit'  # Relative directory in working workspace
os.makedirs(DATA_DIR, exist_ok=True)  # Creates 'streamlit' folder if missing

MODEL_FILE = os.path.join(DATA_DIR, 'rf_model.pkl')
joblib.dump(pipeline, MODEL_FILE)

print(f'\nSuccess! Model pipeline saved to: {os.path.abspath(MODEL_FILE)}')

--- Model Evaluation ---
R² Score: 0.9834
Mean Absolute Error (MAE): 0.68%

Success! Model pipeline saved to: c:\scripts\codeInst_projects\Project2\EV_Battery_Failure_Prediction\streamlit\rf_model.pkl
